In [1]:
from torch.utils.data import Dataset

class CharTokenizer():
    def __init__(self):
        self.vocab = None
        self.char2idx_map = None
        self.idx2char_map = None

    def preprocess(self,text:str):
        text = text.lower()
        chars = text
        if len(text)>1:
            chars = list(text)
        return chars

    def char2idx(self,text:str):
        chars = self.preprocess(text)
        self.vocab = sorted(set(chars))
        self.char2idx_map  = {char: idx for idx, char in enumerate(self.vocab)}
        self.idx2char_map = {idx: char for char, idx in self.char2idx_map.items()}
        return self.vocab, self.char2idx_map, self.idx2char_map
    
    def idx2char(self,idx:int):
        return self.idx2char_map[idx]

    def seq_generator(self,text:str,window:int):
        text_pro = self.preprocess(text)
        vocab, char2idx_map, idx2char_map = self.char2idx(text)
        seqs = []
        for idx in range(len(text_pro) - window):
            input = text_pro[idx:idx+window]
            output = text_pro[idx+1:idx+window+1]
            input_idx = [char2idx_map[char] for char in input]
            output_idx = [char2idx_map[char] for char in output]
            seq = (input_idx,output_idx)
            seqs.append(seq)
        return seqs

class CustomDataset(Dataset):
    def __init__(self,x,y,tranform=None):
        super().__init__()
        self.x = x
        self.y = y
        self.transform = tranform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        x = self.x[index]
        y = self.y[index]
        if self.transform:
            x = self.transform(x)
        return x, y

class DataGenerator():
    @staticmethod
    def generate_pair(pairs:list[tuple]):
        x = []
        y = []
        for input,output in pairs:
            x.append(input)
            y.append(output)
        return x, y

In [3]:
import torch
import math
from torch import nn


class PositionEncoding(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self,seq_len:int,hidden_state:int)->torch.tensor:
        pos = torch.zeros(seq_len,hidden_state)
        max_pos = torch.arange(seq_len).unsqueeze(1)
        fun = torch.exp(torch.arange(0, hidden_state, 2).float() * (-math.log(10000.0) / hidden_state))
        angle = max_pos * fun
        pos[:,0::2] = torch.sin(angle)
        pos[:,1::2] = torch.cos(angle)
        return pos


class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_states,no_of_heads):
        super().__init__()
        self.hidden = hidden_states
        self.heads = no_of_heads
        self.head_dim = hidden_states // no_of_heads
        self.q = nn.Linear(self.hidden,self.hidden,bias=False)
        self.k = nn.Linear(self.hidden,self.hidden,bias=False)
        self.v = nn.Linear(self.hidden,self.hidden,bias=False)
        self.w = nn.Linear(self.hidden,self.hidden,bias=False)

    def forward(self,source,mask=None):
        batch,seq_len,_=source.shape
        q = self.q(source)
        k = self.k(source)
        v = self.v(source)
        q_h = q.reshape(batch,seq_len,self.heads,-1).transpose(1,2)
        k_h = k.reshape(batch,seq_len,self.heads,-1).transpose(1,2)
        v_h = v.reshape(batch,seq_len,self.heads,-1).transpose(1,2)
        k_t = torch.transpose(k_h,-2,-1)
        scores = (q_h@k_t) / math.sqrt(self.head_dim)
        if mask is not None:
            mask_matrix = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
            scores = scores.masked_fill(mask_matrix,float('-inf'))
        pred_q_k = torch.softmax(scores,dim=-1)
        output = (pred_q_k @ v_h)
        output = torch.transpose(output,1,2).reshape(batch,seq_len,-1)
        final_output = self.w(output)
        return final_output , pred_q_k

class SingleGPT(nn.Module):
    def __init__(self, hidden_states,no_of_heads=8):
        super().__init__()
        self.hidden_states = hidden_states
        self.no_of_heads = no_of_heads
        self.norm1 = nn.LayerNorm(hidden_states)
        self.norm2 = nn.LayerNorm(hidden_states)
        self.masked_multi_head = MultiHeadAttention(hidden_states=self.hidden_states,no_of_heads=self.no_of_heads)
        self.l1 = nn.Linear(hidden_states,4*hidden_states,bias=True)
        self.l2 = nn.Linear(4*hidden_states,hidden_states,bias=True)

    def forward(self,source:torch.tensor,mask=None):
        z_dash      = source + self.masked_multi_head(self.norm1(source), mask)[0]
        final_output = z_dash + self.l2(torch.relu(self.l1(self.norm2(z_dash))))
        return final_output


class MiniGPT(nn.Module):
    def __init__(self,vocab_size ,no_of_components,hidden_states, no_of_heads,mask=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_states)
        self.out = nn.Linear(hidden_states, vocab_size)
        self.no_of_components = no_of_components
        self.hidden_states = hidden_states
        self.no_of_heads = no_of_heads
        self.mask = mask
        self.pos_enc = PositionEncoding()
        self.gpts = nn.ModuleList()
        for _ in range(self.no_of_components):
            self.gpts.append(SingleGPT(self.hidden_states,no_of_heads))

    def forward(self,source: torch.tensor):
        source = self.embedding(source)
        _,seq_len,_ = source.shape
        positional_encoding = self.pos_enc(seq_len,self.hidden_states)
        source = source+positional_encoding
        for gpt in self.gpts:
            source = gpt(source,mask=self.mask)
        source = self.out(source)
        return source

    def evaluate(self, dataloader, loss_fn):
        self.eval()

        device = next(self.parameters()).device

        total_loss = 0.0
        total_tokens = 0

        with torch.no_grad():
            for x, y in dataloader:
                x = x.to(device).long()
                y = y.to(device).long()

                logits = self(x)

                logits = logits.reshape(-1, logits.size(-1))
                targets = y.reshape(-1)

                loss = loss_fn(logits, targets)

                total_loss += loss.item() * targets.numel()
                total_tokens += targets.numel()

        return total_loss / total_tokens


    def fit(self, train_loader, val_loader, epochs, loss_fn, optimizer):
        self.train()
        device = next(self.parameters()).device
        history = {
            "train_loss": [],
            "val_loss": []
        }

        for epoch in range(epochs):
            total_loss = 0.0
            total_tokens = 0
            for x, y in train_loader:

                x = x.to(device).long()
                y = y.to(device).long()

                optimizer.zero_grad()

                # Forward pass
                logits = self(x)

                logits = logits.reshape(-1, logits.size(-1))
                targets = y.reshape(-1)

                # Calculate loss
                loss = loss_fn(logits, targets)

                # Backpropagation
                loss.backward()

                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(
                    self.parameters(),
                    max_norm=1.0
                )

                optimizer.step()

                total_loss += loss.item() * targets.numel()
                total_tokens += targets.numel()

            # Average training loss
            train_loss = total_loss / total_tokens

            # Validation loss
            val_loss = self.evaluate(val_loader, loss_fn)

            # Store history
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)

            print(
                f"Epoch [{epoch + 1}/{epochs}] | "
                f"Train Loss: {train_loss:.4f} | "
                f"Val Loss: {val_loss:.4f}"
            )

        return history

In [4]:
import ssl
import urllib.request

import certifi

urls = [
    "https://www.gutenberg.org/files/11/11-0.txt",
    "https://www.gutenberg.org/files/1342/1342-0.txt",
    "https://www.gutenberg.org/files/84/84-0.txt",
    "https://www.gutenberg.org/files/1661/1661-0.txt",
]


def load_training_text(urls=urls):
    context = ssl.create_default_context(cafile=certifi.where())
    texts = []
    for url in urls:
        response = urllib.request.urlopen(url, context=context, timeout=30)
        text = response.read().decode("utf-8-sig")
        texts.append(text)
    training_text = "\n\n".join(texts) + "<|endoftext|>"
    return training_text

In [ ]:
data = load_training_text()
seq_gen = CharTokenizer()
data_gen = DataGenerator()
seq = seq_gen.preprocess(data)
x,y = data_gen.generate_pair(seq)

TypeError: CharTokenizer.preprocess() takes 2 positional arguments but 3 were given

: 